# LogicTreeNet scale S: straight-through fine-tune (Kaggle)

**Goal:** close the gap between the soft network (~57.5% on CIFAR-10) and the discretized
network that a pure binary net would actually run (~50.6%).

**Starting point:** the control run (run 7): 57.69% soft / 50.56% hard on the test set.

**Two arms, same everything except one flag:**

| Arm | Forward pass during training | What it tests |
|---|---|---|
| A: `--ste` | discretized network (argmax gate per neuron); gradients flow through the soft gate probabilities (straight-through estimator) | does training on the real discrete network help? |
| B: normal | soft network, as in all earlier runs | does 20 more epochs alone help? |

Both: 20 epochs from the control model's best checkpoint, AdamW, lr 0.005, no weight decay,
evaluated every epoch. Each arm takes about 35-40 minutes on a T4, so about 75 minutes total.
If A beats B on hard accuracy by more than ~3 points (the run-to-run noise we measured),
the improvement comes from the method, not from extra training.

**Expect arm A's soft accuracy to drop, possibly a lot.** Arm A trains only the discretized
network, so the soft network stops being what's optimized. Only hard accuracy matters for
arm A. In a small CPU test (k=8 model, 3 epochs), straight-through fine-tuning raised hard
validation accuracy from 27.6% to 36.5% while the normal fine-tune reached 28.2%, and soft
accuracy fell from 43% to 15%.

## This version trains the baseline first

Use this if the control run's checkpoint wasn't saved. It first retrains the control model
(run 7's exact settings: 200 epochs, weight decay 0.002, ~5.5 hours on a T4), then runs both
fine-tune arms from it (~75 min). Total ~7 hours, inside Kaggle's 12-hour limit.

**Click Save Version -> Save & Run All (Commit)** instead of running interactively, so it
keeps running with the browser closed and the checkpoints are kept when it finishes.
If it stops partway, attach that version's output under Input and run again: every stage
resumes from its own checkpoint.

## Setup

Installs the `datasets` library (used to fetch CIFAR-10 from a fast HuggingFace mirror
instead of torchvision's default, often-slow download source).

**Before running:** in the notebook's right sidebar, under *Settings*, toggle **Internet: On**
(off by default on Kaggle — required for `pip install` and the HuggingFace download) and
select a **GPU** accelerator.

In [ ]:
!pip install -q datasets


## Source files

In [ ]:
%%writefile difflogic.py
"""
Differentiable Logic Gate Networks (dense) and Convolutional LogicTreeNet.

References:
  - Petersen et al., "Deep Differentiable Logic Gate Networks", NeurIPS 2022
    (arXiv:2210.08277) -- dense DiffLogicNet.
  - Petersen et al., "Convolutional Differentiable Logic Gate Networks",
    NeurIPS 2024 (arXiv:2411.04732) -- convolutional LogicTreeNet.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# The 16 possible 2-input binary Boolean gates, represented as bilinear
# polynomials of the two (soft, in [0,1]) inputs a, b:
#
#   gate(a, b) = k0 + k1*a + k2*b + k3*a*b
#
# For hard (0/1) inputs this recovers the exact truth table.  We build the
# truth tables programmatically (index i -> 4-bit truth table over
# (a,b) in {(0,0),(0,1),(1,0),(1,1)}) rather than hand-typing them, since a
# hand-typed table is an easy place to introduce an off-by-one gate-index bug.
# ---------------------------------------------------------------------------

def _bit(i, k):
    return (i >> k) & 1


# _ALL_TRUTH_TABLES[i] = (out(0,0), out(0,1), out(1,0), out(1,1)) for gate i
_ALL_TRUTH_TABLES = [
    tuple(_bit(i, k) for k in range(4)) for i in range(16)
]


def _truth_table_to_bilinear(tt):
    # tt = (f00, f01, f10, f11); solve for k0,k1,k2,k3 s.t.
    #   f(a,b) = k0 + k1*a + k2*b + k3*a*b
    f00, f01, f10, f11 = tt
    k0 = f00
    k1 = f10 - f00
    k2 = f01 - f00
    k3 = f11 - f10 - f01 + f00
    return (float(k0), float(k1), float(k2), float(k3))


GATE_COEFFS = torch.tensor(
    [_truth_table_to_bilinear(tt) for tt in _ALL_TRUTH_TABLES],
    dtype=torch.float32,
)  # (16, 4)

_NAME_FROM_TT = {
    (0, 0, 0, 0): "FALSE",
    (0, 0, 0, 1): "AND",
    (0, 0, 1, 0): "A_AND_NOT_B",
    (0, 0, 1, 1): "A",
    (0, 1, 0, 0): "NOT_A_AND_B",
    (0, 1, 0, 1): "B",
    (0, 1, 1, 0): "XOR",
    (0, 1, 1, 1): "OR",
    (1, 0, 0, 0): "NOR",
    (1, 0, 0, 1): "XNOR",
    (1, 0, 1, 0): "NOT_B",
    (1, 0, 1, 1): "A_OR_NOT_B",
    (1, 1, 0, 0): "NOT_A",
    (1, 1, 0, 1): "NOT_A_OR_B",
    (1, 1, 1, 0): "NAND",
    (1, 1, 1, 1): "TRUE",
}

GATE_NAMES = [_NAME_FROM_TT[tt] for tt in _ALL_TRUTH_TABLES]

_PASS_A_IDX = GATE_NAMES.index("A")
_PASS_B_IDX = GATE_NAMES.index("B")


def gate_probs(logits, sharpness=1.0, ste=False):
    """Softmax over the 16 gates. With ste=True (straight-through estimator) the
    forward value is the one-hot argmax gate -- exactly the discretized network --
    while gradients flow as if it were the softmax."""
    probs = F.softmax(logits * sharpness, dim=-1)
    if ste:
        hard = F.one_hot(probs.argmax(dim=-1), probs.shape[-1]).to(probs.dtype)
        probs = hard + probs - probs.detach()
    return probs


def gate_forward(logits, a, b, sharpness=1.0, ste=False):
    """logits: (..., 16); a, b: (...,) broadcastable soft values in [0,1]."""
    coeffs = GATE_COEFFS.to(device=logits.device, dtype=logits.dtype)
    probs = gate_probs(logits, sharpness, ste)  # (..., 16)
    k = probs @ coeffs  # (..., 4) -> k0,k1,k2,k3
    k0, k1, k2, k3 = k.unbind(-1)
    return k0 + k1 * a + k2 * b + k3 * a * b


def gate_forward_hard(gate_idx, a, b):
    """gate_idx: (...,) long tensor of chosen gate index; a,b in {0,1}."""
    coeffs = GATE_COEFFS.to(device=gate_idx.device)
    k = coeffs[gate_idx]  # (..., 4)
    k0, k1, k2, k3 = k.unbind(-1)
    out = k0 + k1 * a + k2 * b + k3 * a * b
    return out.round().clamp(0, 1)


def gate_confidence(logits):
    """Mean max-softmax-probability over all gates in a logits tensor.
    1.0 = fully one-hot (confident); 1/16 = uniform (no preference)."""
    probs = F.softmax(logits, dim=-1)
    return probs.max(dim=-1).values.mean().item()


def _residual_gate_init(out_features, generator, bias=5.0, noise=0.1):
    """Residual/pass-through initialization (paper Sec 3.2, z3=5): biases
    each neuron's gate-choice logits toward the pass-through gate ('A' or
    'B', chosen at random per neuron) so that at init the network behaves
    close to identity, preventing vanishing gradients in deep stacks.
    Without this bias, zero/small-random init gives a near-uniform softmax
    over all 16 gates, whose *averaged* output is a constant independent of
    the inputs -- i.e. zero Jacobian and dead backprop through all but the
    last layer.
    """
    logits = torch.randn(out_features, 16, generator=generator) * noise
    choice = torch.randint(0, 2, (out_features,), generator=generator)
    pass_idx = torch.where(choice == 0, _PASS_A_IDX, _PASS_B_IDX)
    logits[torch.arange(out_features), pass_idx] += bias
    return logits


class LogicLayer(nn.Module):
    """A dense differentiable-logic-gate layer with fixed random wiring."""

    def __init__(self, in_features, out_features, seed=0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        g = torch.Generator().manual_seed(seed)

        idx_a = torch.empty(out_features, dtype=torch.long)
        idx_b = torch.empty(out_features, dtype=torch.long)
        for i in range(out_features):
            a = torch.randint(0, in_features, (1,), generator=g).item()
            b = torch.randint(0, in_features, (1,), generator=g).item()
            tries = 0
            while b == a and in_features > 1 and tries < 100:
                b = torch.randint(0, in_features, (1,), generator=g).item()
                tries += 1
            idx_a[i] = a
            idx_b[i] = b
        self.register_buffer("idx_a", idx_a)
        self.register_buffer("idx_b", idx_b)

        self.gate_logits = nn.Parameter(_residual_gate_init(out_features, g))

    def forward(self, x, hard=False, gate_idx=None, sharpness=1.0, ste=False):
        a = x[..., self.idx_a]
        b = x[..., self.idx_b]
        if hard:
            idx = self.gate_logits.argmax(dim=-1) if gate_idx is None else gate_idx
            return gate_forward_hard(idx, a, b)
        return gate_forward(self.gate_logits, a, b, sharpness=sharpness, ste=ste)


class GroupSum(nn.Module):
    def __init__(self, num_classes, tau=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.tau = tau

    def forward(self, x):
        b = x.shape[0]
        group = x.view(b, self.num_classes, -1).sum(dim=-1)
        return group / self.tau


def binarize(x, thresholds=(0.5,), flatten=True):
    """x: (B, C, H, W) float in [0,1]. Returns thresholded binary channels."""
    chans = [(x >= t).float() for t in thresholds]
    out = torch.cat(chans, dim=1)
    if flatten:
        b = out.shape[0]
        return out.view(b, -1)
    return out


class DiffLogicNet(nn.Module):
    def __init__(self, in_features, width, depth, num_classes, tau=1.0, seed=0):
        super().__init__()
        layers = []
        f_in = in_features
        for i in range(depth):
            layers.append(LogicLayer(f_in, width, seed=seed + i))
            f_in = width
        self.layers = nn.ModuleList(layers)
        self.groupsum = GroupSum(num_classes, tau=tau)

    def forward(self, x, hard=False, sharpness=1.0, ste=False):
        for layer in self.layers:
            x = layer(x, hard=hard, sharpness=sharpness, ste=ste)
        return self.groupsum(x)

    def gate_confidences(self):
        return {f"layer_{i}": gate_confidence(l.gate_logits) for i, l in enumerate(self.layers)}


DIFFLOGICNET_CIFAR = {
    "small": dict(width=12000, depth=4, tau=1 / 0.03, reported=51.27),
    "medium": dict(width=48000, depth=4, tau=1 / 0.03, reported=57.68),
    "large": dict(width=128000, depth=4, tau=1 / 0.03, reported=59.38),
}


class LogicTree(nn.Module):
    """A balanced binary tree of logic gates: n_leaves = 2**depth soft
    inputs are folded pairwise, depth times, down to a single output, for
    each of n_trees independent trees (used as a conv kernel)."""

    def __init__(self, n_trees, depth, seed=0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        self.n_leaves = 2 ** depth
        g = torch.Generator().manual_seed(seed)
        n_gates = self.n_leaves - 1
        self.gate_logits = nn.Parameter(
            _residual_gate_init(n_trees * n_gates, g).view(n_trees, n_gates, 16)
        )

    def forward(self, leaves, hard=False, sharpness=1.0, ste=False):
        # leaves: (..., n_trees, n_leaves)
        x = leaves
        gate_offset = 0
        level_size = self.n_leaves
        while level_size > 1:
            n_pairs = level_size // 2
            a = x[..., 0::2]
            b = x[..., 1::2]
            logits = self.gate_logits[:, gate_offset:gate_offset + n_pairs, :]
            if hard:
                idx = logits.argmax(dim=-1)
                x = gate_forward_hard(idx, a, b)
            else:
                probs = gate_probs(logits, sharpness, ste)
                coeffs = GATE_COEFFS.to(device=logits.device, dtype=logits.dtype)
                k = probs @ coeffs
                k0, k1, k2, k3 = k.unbind(-1)
                x = k0 + k1 * a + k2 * b + k3 * a * b
            gate_offset += n_pairs
            level_size = n_pairs
        return x[..., 0]  # (..., n_trees)


class LogicTreeConv2d(nn.Module):
    """Uses a LogicTree per output channel as a convolutional kernel, via
    F.unfold to extract sliding-window patches.

    channels_per_tree: paper Sec 3.1 -- "each tree observes only 2 (rather
    than up to 8) input channels", which forces spatial comparisons within a
    channel. Each tree picks that many distinct input channels at random,
    then draws its 2**depth leaves (without replacement, when possible) from
    those channels' kernel_size x kernel_size windows. 0/None = draw leaves
    from all input channels. Default 0: on CIFAR-10 scale S, 2-channel trees
    gave the same soft accuracy (57.39% vs 57.40%) but much worse hard
    accuracy (46.23% vs 53.54%) than all-channel trees.
    """

    def __init__(self, in_channels, out_channels, kernel_size, tree_depth, seed=0,
                 channels_per_tree=0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.tree_depth = tree_depth
        self.channels_per_tree = channels_per_tree
        n_leaves = 2 ** tree_depth
        kk = kernel_size * kernel_size
        window = in_channels * kk
        g = torch.Generator().manual_seed(seed)
        leaf_idx = torch.empty(out_channels, n_leaves, dtype=torch.long)
        if not channels_per_tree:
            for c in range(out_channels):
                leaf_idx[c] = torch.randint(0, window, (n_leaves,), generator=g)
        else:
            n_ch = min(channels_per_tree, in_channels)
            offsets = torch.arange(kk)
            for c in range(out_channels):
                chans = torch.randperm(in_channels, generator=g)[:n_ch]
                # F.unfold lays patches out channel-major: index = ch * kk + pos
                cand = (chans[:, None] * kk + offsets[None, :]).reshape(-1)
                for _ in range(100):
                    if cand.numel() >= n_leaves:
                        pick = torch.randperm(cand.numel(), generator=g)[:n_leaves]
                    else:
                        pick = torch.randint(0, cand.numel(), (n_leaves,), generator=g)
                    # redraw in the rare case every leaf landed in one channel
                    if len(set((cand[pick] // kk).tolist())) == n_ch:
                        break
                leaf_idx[c] = cand[pick]
        self.register_buffer("leaf_idx", leaf_idx)
        self.tree = LogicTree(out_channels, tree_depth, seed=seed + 1)

    def forward(self, x, hard=False, sharpness=1.0, ste=False):
        b, c, h, w = x.shape
        patches = F.unfold(x, kernel_size=self.kernel_size, padding=self.kernel_size // 2)
        # patches: (B, window, L)
        l = patches.shape[-1]
        patches = patches.transpose(1, 2)  # (B, L, window)
        # gather leaves: (B, L, out_channels, n_leaves)
        leaves = patches[:, :, self.leaf_idx]
        out = self.tree(leaves, hard=hard, sharpness=sharpness, ste=ste)  # (B, L, out_channels)
        out = out.transpose(1, 2).contiguous()  # (B, out_channels, L)
        out_h = h  # padding = kernel_size//2 keeps spatial size
        out_w = w
        return out.view(b, self.out_channels, out_h, out_w)


class OrPool2d(nn.Module):
    """Logical-OR pooling: soft = probabilistic-sum-OR = 1 - prod(1-x) over
    the window; hard = max-pool (since OR of bits == max of bits)."""

    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x, hard=False):
        if hard:
            return F.max_pool2d(x, self.kernel_size, self.stride)
        # Product over the window written as explicit multiplies of strided slices.
        # (torch.prod on CUDA is JIT-compiled via NVRTC, which is broken on some
        # Colab images: "failed to open libnvrtc-builtins.so".)
        b, c, h, w = x.shape
        k, s = self.kernel_size, self.stride
        out_h = (h - k) // s + 1
        out_w = (w - k) // s + 1
        keep = None
        for i in range(k):
            for j in range(k):
                sl = x[:, :, i:i + s * (out_h - 1) + 1:s, j:j + s * (out_w - 1) + 1:s]
                keep = (1 - sl) if keep is None else keep * (1 - sl)
        return 1 - keep


class LogicTreeNet(nn.Module):
    """Convolutional differentiable logic gate network (paper: LogicTreeNet).

    Paper Appendix A.1.1: after the conv+pool blocks, the head is exactly
    3 dense LogicLayers of width 1280k -> 640k -> 320k, where k =
    channels[0] (the base scale, e.g. 32 for scale S). This is NOT a free
    architectural choice.
    """

    def __init__(self, in_channels, in_size, channels, num_classes,
                 tree_depth=3, kernel_size=3, output_gate_factor=1,
                 tau=20.0, seed=0, channels_per_tree=0):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = in_channels
        size = in_size
        for i, c_out in enumerate(channels):
            self.blocks.append(
                LogicTreeConv2d(c_in, c_out, kernel_size, tree_depth, seed=seed + i,
                                channels_per_tree=channels_per_tree)
            )
            self.pools.append(OrPool2d(2, 2))
            c_in = c_out
            size = size // 2
        flat_dim = c_in * size * size

        # Paper Appendix A.1.1: k = channels[0]; head is exactly 3 dense
        # LogicLayers of width 1280k -> 640k -> 320k (not a free choice).
        k = channels[0]
        ox = output_gate_factor
        head_widths = [1280 * k * ox, 640 * k * ox, 320 * k * ox]
        assert head_widths[-1] % num_classes == 0, (
            f"final head width {head_widths[-1]} must be divisible by num_classes={num_classes}"
        )

        head_layers = []
        f_in = flat_dim
        for i, w in enumerate(head_widths):
            head_layers.append(LogicLayer(f_in, w, seed=seed + 100 + i))
            f_in = w
        self.head = nn.ModuleList(head_layers)
        self.groupsum = GroupSum(num_classes, tau=tau)
        self.flat_dim = flat_dim

    def forward(self, x, hard=False, sharpness=1.0, ste=False):
        # ste=True: forward pass is exactly the discretized network (one-hot gates
        # on binary inputs give binary outputs, and soft OR-pooling of bits is
        # exact OR), but gradients flow through the gate softmaxes.
        for block, pool in zip(self.blocks, self.pools):
            x = block(x, hard=hard, sharpness=sharpness, ste=ste)
            x = pool(x, hard=hard)
        x = x.flatten(1)
        for layer in self.head:
            x = layer(x, hard=hard, sharpness=sharpness, ste=ste)
        return self.groupsum(x)

    def gate_confidences(self):
        confs = {}
        for i, block in enumerate(self.blocks):
            confs[f"conv_block_{i}"] = gate_confidence(block.tree.gate_logits)
        for i, layer in enumerate(self.head):
            confs[f"head_{i}"] = gate_confidence(layer.gate_logits)
        return confs


# Paper Table 6 (CIFAR-10). Scale B removed: needs unimplemented 5-bit
# input + edge/curvature-detector preprocessing (paper Sec 3.5).
LOGICTREENET_CIFAR = {
    "S": dict(channels=(32, 128, 512, 1024), tau=20.0, reported=60.38),
    "M": dict(channels=(256, 1024, 4096, 8192), tau=40.0, reported=71.01),
}


In [ ]:
%%writefile test_difflogic.py
"""Test suite for difflogic.py / train.py. Run: python test_difflogic.py"""

import sys
import torch
import torch.nn.functional as F

sys.path.insert(0, ".")

from difflogic import (
    GATE_COEFFS, GATE_NAMES, _ALL_TRUTH_TABLES,
    gate_forward, gate_forward_hard, gate_confidence,
    _residual_gate_init, _PASS_A_IDX, _PASS_B_IDX,
    LogicLayer, GroupSum, binarize, DiffLogicNet, DIFFLOGICNET_CIFAR,
    LogicTree, LogicTreeConv2d, OrPool2d, LogicTreeNet, LOGICTREENET_CIFAR,
)
from train import sharpness_at, weight_decay_at

passed = 0
failed = 0


def check(name, cond):
    global passed, failed
    if cond:
        passed += 1
    else:
        failed += 1
        print(f"FAILED: {name}")


# ---------------------------------------------------------------------------
# Gate truth tables / bilinear coefficients
# ---------------------------------------------------------------------------

corners = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]

for i in range(16):
    tt = _ALL_TRUTH_TABLES[i]
    k = GATE_COEFFS[i]
    for j, (a, b) in enumerate(corners):
        val = k[0] + k[1] * a + k[2] * b + k[3] * a * b
        check(f"gate {i} ({GATE_NAMES[i]}) truth-table corner {j}", abs(val.item() - tt[j]) < 1e-6)

check("16 gate names, all unique", len(set(GATE_NAMES)) == 16)
check("AND gate correct", GATE_NAMES[_ALL_TRUTH_TABLES.index((0, 0, 0, 1))] == "AND")
check("OR gate correct", GATE_NAMES[_ALL_TRUTH_TABLES.index((0, 1, 1, 1))] == "OR")
check("XOR gate correct", GATE_NAMES[_ALL_TRUTH_TABLES.index((0, 1, 1, 0))] == "XOR")
check("pass-A gate found", GATE_NAMES[_PASS_A_IDX] == "A")
check("pass-B gate found", GATE_NAMES[_PASS_B_IDX] == "B")

# bilinear-softmax-collapse identity vs explicit 16-gate averaging
torch.manual_seed(0)
logits = torch.randn(5, 16)
a = torch.rand(5)
b = torch.rand(5)
fast = gate_forward(logits, a, b, sharpness=1.0)
probs = F.softmax(logits, dim=-1)
slow = torch.zeros(5)
for i in range(16):
    tt = _ALL_TRUTH_TABLES[i]
    k = GATE_COEFFS[i]
    slow += probs[:, i] * (k[0] + k[1] * a + k[2] * b + k[3] * a * b)
check("bilinear-softmax-collapse matches explicit 16-gate average", torch.allclose(fast, slow, atol=1e-5))

# hard/soft agreement at extreme one-hot logits
onehot = torch.full((4, 16), -100.0)
for i, idx in enumerate([_ALL_TRUTH_TABLES.index((0, 0, 0, 1)),  # AND
                          _ALL_TRUTH_TABLES.index((0, 1, 1, 1)),  # OR
                          _ALL_TRUTH_TABLES.index((0, 1, 1, 0)),  # XOR
                          _PASS_A_IDX]):
    onehot[i, idx] = 100.0
aa = torch.tensor([1.0, 0.0, 1.0, 1.0])
bb = torch.tensor([1.0, 0.0, 1.0, 0.0])
soft_out = gate_forward(onehot, aa, bb, sharpness=1.0)
idx_hard = onehot.argmax(dim=-1)
hard_out = gate_forward_hard(idx_hard, aa, bb)
check("hard/soft agree at one-hot extreme logits", torch.allclose(soft_out.round(), hard_out, atol=1e-3))

check("gate_confidence near 1.0 for one-hot logits", gate_confidence(onehot) > 0.99)
uniform_logits = torch.zeros(10, 16)
check("gate_confidence near 1/16 for uniform logits", abs(gate_confidence(uniform_logits) - 1 / 16) < 1e-3)

# ---------------------------------------------------------------------------
# Residual init
# ---------------------------------------------------------------------------

g = torch.Generator().manual_seed(0)
ri = _residual_gate_init(1000, g, bias=5.0, noise=0.1)
probs_ri = F.softmax(ri, dim=-1)
maxp = probs_ri.max(dim=-1).values
check("residual init: most neurons strongly favor pass-through gate", (maxp > 0.5).float().mean().item() > 0.8)

# ---------------------------------------------------------------------------
# LogicLayer
# ---------------------------------------------------------------------------

layer = LogicLayer(20, 30, seed=1)
check("LogicLayer wiring in range", (layer.idx_a < 20).all() and (layer.idx_b < 20).all())
check("LogicLayer wiring mostly distinct a!=b", (layer.idx_a != layer.idx_b).float().mean().item() > 0.9)
x = torch.rand(4, 20, requires_grad=True)
out = layer(x, hard=False)
check("LogicLayer output shape", out.shape == (4, 30))
loss = out.sum()
loss.backward()
check("LogicLayer gate_logits gradient nonzero", layer.gate_logits.grad.norm().item() > 1e-12)
check("LogicLayer input gradient nonzero", x.grad.norm().item() > 1e-12)

# same seed -> same wiring (determinism)
layer2 = LogicLayer(20, 30, seed=1)
check("LogicLayer wiring deterministic across construction w/ same seed",
      torch.equal(layer.idx_a, layer2.idx_a) and torch.equal(layer.idx_b, layer2.idx_b))

# ---------------------------------------------------------------------------
# GroupSum / binarize
# ---------------------------------------------------------------------------

gs = GroupSum(num_classes=5, tau=2.0)
xin = torch.ones(3, 20)
gout = gs(xin)
check("GroupSum shape", gout.shape == (3, 5))
check("GroupSum values correct", torch.allclose(gout, torch.full((3, 5), 2.0)))

img = torch.rand(2, 3, 8, 8)
bz = binarize(img, thresholds=(0.25, 0.5, 0.75), flatten=False)
check("binarize channel count", bz.shape[1] == 3 * 3)
check("binarize output is 0/1", set(bz.unique().tolist()).issubset({0.0, 1.0}))
bz_flat = binarize(img, thresholds=(0.25, 0.5, 0.75), flatten=True)
check("binarize flatten shape", bz_flat.shape == (2, 3 * 3 * 8 * 8))

# ---------------------------------------------------------------------------
# DiffLogicNet
# ---------------------------------------------------------------------------

net = DiffLogicNet(in_features=40, width=64, depth=3, num_classes=4, tau=1.0, seed=0)
xin = torch.rand(6, 40, requires_grad=True)
out = net(xin, hard=False)
check("DiffLogicNet output shape", out.shape == (6, 4))
out.sum().backward()
for i, l in enumerate(net.layers):
    check(f"DiffLogicNet layer {i} gradient nonzero (no dead layers)", l.gate_logits.grad.norm().item() > 1e-12)
check("DiffLogicNet input gradient nonzero", xin.grad.norm().item() > 1e-12)

hard_out = net(torch.randint(0, 2, (6, 40)).float(), hard=True)
check("DiffLogicNet hard forward finite", torch.isfinite(hard_out).all())

confs = net.gate_confidences()
check("DiffLogicNet gate_confidences has one entry per layer", len(confs) == len(net.layers))

for scale, cfg in DIFFLOGICNET_CIFAR.items():
    check(f"DIFFLOGICNET_CIFAR[{scale}] has required keys",
          {"width", "depth", "tau", "reported"}.issubset(cfg.keys()))

# ---------------------------------------------------------------------------
# LogicTree
# ---------------------------------------------------------------------------

tree = LogicTree(n_trees=5, depth=3, seed=2)
leaves = torch.rand(4, 5, 8, requires_grad=True)
tout = tree(leaves, hard=False)
check("LogicTree output shape", tout.shape == (4, 5))
tout.sum().backward()
check("LogicTree gate_logits gradient nonzero", tree.gate_logits.grad.norm().item() > 1e-12)
check("LogicTree input gradient nonzero", leaves.grad.norm().item() > 1e-12)

hard_leaves = torch.randint(0, 2, (4, 5, 8)).float()
tout_hard = tree(hard_leaves, hard=True)
check("LogicTree hard forward is 0/1", set(tout_hard.unique().tolist()).issubset({0.0, 1.0}))

# ---------------------------------------------------------------------------
# LogicTreeConv2d: vectorized vs naive loop equivalence
# ---------------------------------------------------------------------------

conv = LogicTreeConv2d(in_channels=2, out_channels=3, kernel_size=3, tree_depth=2, seed=3)
ximg = torch.rand(2, 2, 6, 6, requires_grad=True)
cout = conv(ximg, hard=False)
check("LogicTreeConv2d output shape", cout.shape == (2, 3, 6, 6))
cout.sum().backward()
check("LogicTreeConv2d gate_logits gradient nonzero", conv.tree.gate_logits.grad.norm().item() > 1e-12)
check("LogicTreeConv2d input gradient nonzero", ximg.grad.norm().item() > 1e-12)

# naive per-pixel loop check (small case, no grad)
with torch.no_grad():
    conv2 = LogicTreeConv2d(in_channels=2, out_channels=3, kernel_size=3, tree_depth=2, seed=3)
    conv2.load_state_dict(conv.state_dict())
    x_small = torch.rand(1, 2, 5, 5)
    out_vec = conv2(x_small, hard=False)
    padded = F.pad(x_small, (1, 1, 1, 1))
    out_naive = torch.zeros(1, 3, 5, 5)
    window = 2 * 3 * 3
    for hh in range(5):
        for ww in range(5):
            patch = padded[0, :, hh:hh + 3, ww:ww + 3].reshape(-1)  # (window,)
            leaves_px = patch[conv2.leaf_idx]  # (out_channels, n_leaves)
            val = conv2.tree(leaves_px.unsqueeze(0), hard=False)  # (1, out_channels)
            out_naive[0, :, hh, ww] = val[0]
    check("LogicTreeConv2d vectorized matches naive per-pixel loop", torch.allclose(out_vec, out_naive, atol=1e-4))

# ---------------------------------------------------------------------------
# OrPool2d
# ---------------------------------------------------------------------------

pool = OrPool2d(2, 2)
xp = torch.rand(2, 3, 4, 4)
soft_p = pool(xp, hard=False)
check("OrPool2d soft output shape", soft_p.shape == (2, 3, 2, 2))
# manual formula check on one window
manual = 1 - (1 - xp[0, 0, 0, 0]) * (1 - xp[0, 0, 0, 1]) * (1 - xp[0, 0, 1, 0]) * (1 - xp[0, 0, 1, 1])
check("OrPool2d soft formula matches manual 1-prod(1-x)", abs(soft_p[0, 0, 0, 0].item() - manual.item()) < 1e-5)

hard_p = pool(xp, hard=True)
maxp_manual = F.max_pool2d(xp, 2, 2)
check("OrPool2d hard matches max_pool2d", torch.allclose(hard_p, maxp_manual))

xnc = torch.rand(2, 4, 4, 3).permute(0, 3, 1, 2)  # non-contiguous (channels-last) input
check("OrPool2d soft works on non-contiguous input and matches contiguous result",
      torch.allclose(pool(xnc, hard=False), pool(xnc.contiguous(), hard=False)))
check("LogicTreeConv2d output is contiguous", conv(torch.rand(1, 2, 6, 6)).is_contiguous())

# ---------------------------------------------------------------------------
# LogicTreeNet (full model)
# ---------------------------------------------------------------------------

model = LogicTreeNet(in_channels=3, in_size=8, channels=(4, 8), num_classes=5,
                      tree_depth=2, kernel_size=3, tau=1.0, seed=0)
xin = torch.rand(2, 3, 8, 8, requires_grad=True)
out = model(xin, hard=False)
check("LogicTreeNet output shape", out.shape == (2, 5))
out.sum().backward()
for i, block in enumerate(model.blocks):
    check(f"LogicTreeNet conv block {i} gradient nonzero", block.tree.gate_logits.grad.norm().item() > 1e-12)
for i, layer in enumerate(model.head):
    check(f"LogicTreeNet head layer {i} gradient nonzero", layer.gate_logits.grad.norm().item() > 1e-12)
check("LogicTreeNet input gradient nonzero", xin.grad.norm().item() > 1e-12)

hard_in = torch.randint(0, 2, (2, 3, 8, 8)).float()
hard_out = model(hard_in, hard=True)
check("LogicTreeNet hard forward finite", torch.isfinite(hard_out).all())

confs = model.gate_confidences()
check("LogicTreeNet gate_confidences count", len(confs) == len(model.blocks) + len(model.head))

# head width taper check: 1280k -> 640k -> 320k relative ratios (k=channels[0])
k = 4
expected_widths = [1280 * k, 640 * k, 320 * k]
actual_widths = [l.out_features for l in model.head]
check("LogicTreeNet head widths match paper taper 1280k/640k/320k",
      actual_widths == expected_widths)

for scale, cfg in LOGICTREENET_CIFAR.items():
    check(f"LOGICTREENET_CIFAR[{scale}] has required keys",
          {"channels", "tau", "reported"}.issubset(cfg.keys()))
check("LOGICTREENET_CIFAR scale B removed (needs unimplemented preprocessing)", "B" not in LOGICTREENET_CIFAR)
check("LOGICTREENET_CIFAR M channels = 256,1024,4096,8192", LOGICTREENET_CIFAR["M"]["channels"] == (256, 1024, 4096, 8192))
check("LOGICTREENET_CIFAR S channels = 32,128,512,1024", LOGICTREENET_CIFAR["S"]["channels"] == (32, 128, 512, 1024))

s_params = sum(p.numel() for p in
                LogicTreeNet(in_channels=3, in_size=32, num_classes=10, tree_depth=3, kernel_size=3,
                             **{k: v for k, v in LOGICTREENET_CIFAR["S"].items() if k != "reported"}).parameters())
m_params = sum(p.numel() for p in
                LogicTreeNet(in_channels=3, in_size=32, num_classes=10, tree_depth=3, kernel_size=3,
                             **{k: v for k, v in LOGICTREENET_CIFAR["M"].items() if k != "reported"}).parameters())
check("LogicTreeNet scale M has more params than scale S", m_params > s_params)

# ---------------------------------------------------------------------------
# channels_per_tree (paper Sec 3.1: each tree observes only 2 input channels)
# ---------------------------------------------------------------------------

kk = 9
c2 = LogicTreeConv2d(in_channels=16, out_channels=50, kernel_size=3, tree_depth=3, seed=5, channels_per_tree=2)
chans_per_row = [len(set((row // kk).tolist())) for row in c2.leaf_idx]
check("channels_per_tree=2: every tree reads from exactly 2 channels", all(n == 2 for n in chans_per_row))
check("channels_per_tree=2: leaves within a tree are distinct",
      all(len(set(row.tolist())) == 8 for row in c2.leaf_idx))
check("channels_per_tree=2: leaf indices in range", int(c2.leaf_idx.max()) < 16 * kk and int(c2.leaf_idx.min()) >= 0)
check("channels_per_tree=2: different trees use different channel pairs",
      len({tuple(sorted(set((row // kk).tolist()))) for row in c2.leaf_idx}) > 10)

c_all = LogicTreeConv2d(in_channels=16, out_channels=50, kernel_size=3, tree_depth=3, seed=5, channels_per_tree=0)
check("channels_per_tree=0: trees span more than 2 channels (old all-channel behavior)",
      max(len(set((row // kk).tolist())) for row in c_all.leaf_idx) > 2)

c1 = LogicTreeConv2d(in_channels=1, out_channels=4, kernel_size=3, tree_depth=3, seed=0, channels_per_tree=2)
check("channels_per_tree with a single input channel still works",
      c1(torch.rand(2, 1, 5, 5)).shape == (2, 4, 5, 5))

c_small = LogicTreeConv2d(in_channels=3, out_channels=4, kernel_size=1, tree_depth=3, seed=0, channels_per_tree=2)
check("channels_per_tree with fewer candidates than leaves (1x1 kernel) still works",
      c_small(torch.rand(2, 3, 4, 4)).shape == (2, 4, 4, 4)
      and all(len(set((row // 1).tolist())) <= 2 for row in c_small.leaf_idx))

net_cpt = LogicTreeNet(in_channels=3, in_size=8, channels=(4, 8), num_classes=5,
                       tree_depth=2, kernel_size=3, tau=1.0, seed=0)
net_cpt2 = LogicTreeNet(in_channels=3, in_size=8, channels=(4, 8), num_classes=5,
                        tree_depth=2, kernel_size=3, tau=1.0, seed=0, channels_per_tree=2)
check("LogicTreeNet passes channels_per_tree=2 through to every conv block",
      all(b.channels_per_tree == 2 for b in net_cpt2.blocks))
check("LogicTreeNet defaults to channels_per_tree=0 (all channels) in every conv block",
      all(b.channels_per_tree == 0 for b in net_cpt.blocks))
xin = torch.rand(2, 3, 8, 8, requires_grad=True)
net_cpt(xin).sum().backward()
check("LogicTreeNet (default wiring): gradient reaches every conv block",
      all(b.tree.gate_logits.grad.norm().item() > 1e-12 for b in net_cpt.blocks))

# ---------------------------------------------------------------------------
# straight-through (ste) training mode
# ---------------------------------------------------------------------------

from difflogic import gate_probs
lg = torch.randn(7, 16, requires_grad=True)
pr = gate_probs(lg, ste=True)
check("gate_probs(ste=True) forward is exactly one-hot",
      torch.allclose(pr.detach(), F.one_hot(lg.argmax(-1), 16).float(), atol=1e-6))
(pr * torch.randn(7, 16)).sum().backward()
check("gate_probs(ste=True) passes gradients to logits", lg.grad.norm().item() > 1e-8)

torch.manual_seed(1)
ste_net = LogicTreeNet(in_channels=9, in_size=16, channels=(4, 8), num_classes=10,
                       tree_depth=3, kernel_size=3, tau=5.0, seed=3)
for prm in ste_net.parameters():
    prm.data += torch.randn_like(prm) * 3  # move argmaxes off the pass-through init
xb = binarize(torch.rand(6, 3, 16, 16), (0.25, 0.5, 0.75), flatten=False)
with torch.no_grad():
    check("LogicTreeNet ste=True forward equals the hard (discretized) network exactly",
          torch.allclose(ste_net(xb, ste=True), ste_net(xb, hard=True), atol=1e-5))
ste_net.zero_grad()
ste_net(xb, ste=True).sum().backward()
check("LogicTreeNet ste=True: gradient reaches every conv block and head layer",
      all(b.tree.gate_logits.grad.norm().item() > 1e-12 for b in ste_net.blocks)
      and all(l.gate_logits.grad.norm().item() > 1e-12 for l in ste_net.head))

dn = DiffLogicNet(in_features=48, width=40, depth=3, num_classes=10, tau=2.0, seed=2)
for prm in dn.parameters():
    prm.data += torch.randn_like(prm) * 3
xdn = binarize(torch.rand(5, 3, 4, 4), (0.5,), flatten=True)
with torch.no_grad():
    check("DiffLogicNet ste=True forward equals hard forward",
          torch.allclose(dn(xdn, ste=True), dn(xdn, hard=True), atol=1e-5))

# ---------------------------------------------------------------------------
# sharpness_at / weight_decay_at schedules
# ---------------------------------------------------------------------------

check("sharpness_at off when final_sharpness<=1.0", sharpness_at(50, 100, 1.0, 0.5) == 1.0)
check("sharpness_at held at 1.0 during warmup", sharpness_at(10, 100, 5.0, 0.5) == 1.0)
check("sharpness_at reaches final at last epoch", abs(sharpness_at(99, 100, 5.0, 0.5) - 5.0) < 0.2)
s_mid = sharpness_at(75, 100, 5.0, 0.5)
check("sharpness_at monotonic non-decreasing after warmup",
      sharpness_at(60, 100, 5.0, 0.5) <= s_mid <= sharpness_at(90, 100, 5.0, 0.5))

check("weight_decay_at off (base unchanged) when final_sharpness<=1.0",
      weight_decay_at(50, 100, 0.002, 1.0, 0.5) == 0.002)
check("weight_decay_at held at base during warmup", weight_decay_at(10, 100, 0.002, 5.0, 0.5) == 0.002)
check("weight_decay_at decays to ~0 at end", weight_decay_at(99, 100, 0.002, 5.0, 0.5) < 0.0005)
wd_mid = weight_decay_at(75, 100, 0.002, 5.0, 0.5)
check("weight_decay_at monotonic non-increasing after warmup",
      weight_decay_at(60, 100, 0.002, 5.0, 0.5) >= wd_mid >= weight_decay_at(90, 100, 0.002, 5.0, 0.5))

# ---------------------------------------------------------------------------

print(f"{passed} passed, {failed} failed")
if failed:
    raise SystemExit(1)


In [ ]:
%%writefile train.py
"""CLI trainer for DiffLogicNet (dense) and LogicTreeNet (conv) on CIFAR-10."""

import argparse
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

from difflogic import (
    DiffLogicNet, DIFFLOGICNET_CIFAR,
    LogicTreeNet, LOGICTREENET_CIFAR,
    binarize,
)


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

class _HFCIFAR10(torch.utils.data.Dataset):
    """Wraps the uoft-cs/cifar10 HuggingFace mirror (much faster to fetch
    than torchvision's default download source in many environments)."""

    def __init__(self, split, transform=None):
        from datasets import load_dataset
        self.ds = load_dataset("uoft-cs/cifar10", split=split)
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        img = row["img"]
        label = row["label"]
        if self.transform is not None:
            img = self.transform(img)
        return img, label


def load_cifar10(data_dir, val_split, thresholds, augment=False):
    """45k/5k/10k split. With augment=True the training images (only) get a
    random 32x32 crop from a 4-pixel zero-padded image plus a random
    horizontal flip, applied before binarization. The paper does not mention
    augmentation, so it is off by default."""
    plain = T.Compose([T.ToTensor()])
    train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor()]) if augment else plain
    try:
        train_full = _HFCIFAR10("train", transform=train_tf)
        val_full = _HFCIFAR10("train", transform=plain)
        test_set = _HFCIFAR10("test", transform=plain)
    except Exception as e:
        print(f"HF CIFAR-10 mirror unavailable ({e}); falling back to torchvision download.")
        train_full = torchvision.datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
        val_full = torchvision.datasets.CIFAR10(data_dir, train=True, download=True, transform=plain)
        test_set = torchvision.datasets.CIFAR10(data_dir, train=False, download=True, transform=plain)

    n = len(train_full)
    g = torch.Generator().manual_seed(0)
    perm = torch.randperm(n, generator=g)
    val_idx = perm[:val_split]
    train_idx = perm[val_split:]
    train_set = torch.utils.data.Subset(train_full, train_idx.tolist())
    val_set = torch.utils.data.Subset(val_full, val_idx.tolist())
    return train_set, val_set, test_set


def collate_binarize(batch, thresholds):
    imgs = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    return imgs, labels


def pick_device(name="auto"):
    """auto = CUDA GPU, else Apple-silicon GPU (MPS), else CPU."""
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


# ---------------------------------------------------------------------------
# Model construction
# ---------------------------------------------------------------------------

def build_model(args, in_ch, in_size, n_classes, n_thresh):
    if args.model == "dense":
        cfg = dict(DIFFLOGICNET_CIFAR[args.scale])
        cfg.pop("reported", None)
        if args.tau_override is not None:
            cfg["tau"] = args.tau_override
        in_features = in_ch * n_thresh * in_size * in_size
        return DiffLogicNet(in_features=in_features, num_classes=n_classes, seed=args.seed, **cfg)
    elif args.model == "conv":
        cfg = dict(LOGICTREENET_CIFAR[args.scale])
        cfg.pop("reported", None)
        if args.tau_override is not None:
            cfg["tau"] = args.tau_override
        return LogicTreeNet(
            in_channels=in_ch * n_thresh,
            in_size=in_size,
            num_classes=n_classes,
            tree_depth=args.tree_depth,
            kernel_size=args.kernel_size,
            seed=args.seed,
            channels_per_tree=args.channels_per_tree,
            **cfg,
        )
    else:
        raise ValueError(f"unknown model {args.model}")


# ---------------------------------------------------------------------------
# Annealing schedules
# ---------------------------------------------------------------------------

def sharpness_at(epoch, total_epochs, final_sharpness, warmup_frac):
    """Linear ramp: sharpness=1.0 held during warmup_frac of training, then
    ramps linearly to final_sharpness by the end. Returns 1.0 (off) if
    final_sharpness <= 1.0."""
    if final_sharpness <= 1.0:
        return 1.0
    warmup_epochs = warmup_frac * total_epochs
    if epoch <= warmup_epochs:
        return 1.0
    if total_epochs <= warmup_epochs:
        return final_sharpness
    frac = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    frac = min(max(frac, 0.0), 1.0)
    return 1.0 + frac * (final_sharpness - 1.0)


def weight_decay_at(epoch, total_epochs, base_wd, final_sharpness, warmup_frac):
    """Anneals weight decay to 0 over the same window sharpness ramps up in
    (plain weight decay fights sharpness annealing: it pulls logit
    magnitude toward zero while sharpness tries to grow it). Returns
    base_wd unchanged if final_sharpness <= 1.0 (annealing off)."""
    if final_sharpness <= 1.0:
        return base_wd
    warmup_epochs = warmup_frac * total_epochs
    if epoch <= warmup_epochs:
        return base_wd
    if total_epochs <= warmup_epochs:
        return 0.0
    frac = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    frac = min(max(frac, 0.0), 1.0)
    return base_wd * (1.0 - frac)


# ---------------------------------------------------------------------------
# Eval
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader, thresholds, device, hard=False, sharpness=1.0, flatten=True):
    model.eval()
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        x = binarize(imgs, thresholds=thresholds, flatten=flatten)
        out = model(x, hard=hard, sharpness=sharpness)
        pred = out.argmax(dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.numel()
    model.train()
    return correct / total


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", choices=["dense", "conv"], required=True)
    p.add_argument("--scale", required=True)
    p.add_argument("--tree-depth", type=int, default=3)
    p.add_argument("--kernel-size", type=int, default=3)
    p.add_argument("--thresholds", type=str, default="0.5")
    p.add_argument("--epochs", type=int, default=200)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.02)
    p.add_argument("--weight-decay", type=float, default=0.0)
    p.add_argument("--optimizer", choices=["adam", "adamw"], default="adam")
    p.add_argument("--tau-override", type=float, default=None)
    p.add_argument("--val-split", type=int, default=5000)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--data-dir", type=str, default="./data")
    p.add_argument("--ckpt", type=str, required=True)
    p.add_argument("--log", type=str, required=True)
    p.add_argument("--eval-every", type=int, default=5)
    p.add_argument("--fresh", action="store_true")
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--sharpness-final", type=float, default=1.0)
    p.add_argument("--sharpness-warmup-frac", type=float, default=0.5)
    p.add_argument("--channels-per-tree", type=int, default=0,
                   help="input channels each conv logic tree reads from; 0 = all (default). The paper says 2, "
                        "but that lowered hard accuracy by ~7 pts on scale S here")
    p.add_argument("--device", default="auto", help="auto | cuda | mps | cpu")
    p.add_argument("--ste", action="store_true",
                   help="straight-through training: forward pass uses the discretized (argmax) gates, "
                        "gradients flow through the soft gate probabilities")
    p.add_argument("--init-from", type=str, default=None,
                   help="start from this checkpoint's weights (fresh optimizer, epoch 0) when --ckpt "
                        "doesn't exist yet; used for fine-tuning")
    p.add_argument("--augment", action="store_true",
                   help="random crop (pad 4) + horizontal flip on training images; not in the paper")
    args = p.parse_args()

    device = pick_device(args.device)
    print(f"device: {device}")
    thresholds = tuple(float(t) for t in args.thresholds.split(","))
    n_thresh = len(thresholds)

    train_set, val_set, test_set = load_cifar10(args.data_dir, args.val_split, thresholds, augment=args.augment)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=args.batch_size, shuffle=True,
        num_workers=args.num_workers, drop_last=True,
    )
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=args.num_workers)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=args.num_workers)

    n_classes = 10
    in_ch = 3
    in_size = 32
    flatten = (args.model == "dense")

    model = build_model(args, in_ch, in_size, n_classes, n_thresh).to(device)

    if args.optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    ckpt_path = Path(args.ckpt)
    log_path = Path(args.log)
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    expected_arch = dict(
        model=args.model, scale=args.scale, tree_depth=args.tree_depth,
        kernel_size=args.kernel_size, thresholds=args.thresholds,
        channels_per_tree=(args.channels_per_tree if args.model == "conv" else 0),
    )

    start_epoch = 0
    best_val = -1.0

    if ckpt_path.exists() and not args.fresh:
        state = torch.load(ckpt_path, map_location=device)
        saved_arch = state.get("arch")
        if saved_arch is not None:
            # checkpoints from before channels_per_tree existed used all channels
            saved_arch = {**saved_arch, "channels_per_tree": saved_arch.get("channels_per_tree", 0)}
        if saved_arch != expected_arch:
            raise SystemExit(
                f"Checkpoint architecture mismatch!\n"
                f"  checkpoint arch: {saved_arch}\n"
                f"  requested arch:  {expected_arch}\n"
                f"Pass --fresh to discard the incompatible checkpoint and start over."
            )
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["opt"])
        start_epoch = state["epoch"] + 1
        best_val = state.get("best_val", -1.0)
        print(f"Resumed from {ckpt_path} at epoch {start_epoch}")
    elif args.init_from:
        state = torch.load(args.init_from, map_location=device)
        saved_arch = state.get("arch")
        if saved_arch is not None:
            saved_arch = {**saved_arch, "channels_per_tree": saved_arch.get("channels_per_tree", 0)}
        if saved_arch != expected_arch:
            raise SystemExit(
                f"--init-from checkpoint has a different architecture:\n"
                f"  checkpoint arch: {saved_arch}\n  requested arch:  {expected_arch}")
        model.load_state_dict(state["model"])
        print(f"Initialized weights from {args.init_from} (its epoch {state.get('epoch')}); "
              f"fine-tuning from epoch 0 with a fresh optimizer")
    elif args.fresh:
        print("`--fresh` passed: starting from scratch (ignoring any existing checkpoint).")

    log_f = open(log_path, "a")

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        sharpness = sharpness_at(epoch, args.epochs, args.sharpness_final, args.sharpness_warmup_frac)
        wd_now = weight_decay_at(epoch, args.epochs, args.weight_decay, args.sharpness_final, args.sharpness_warmup_frac)
        for g in opt.param_groups:
            g["weight_decay"] = wd_now

        model.train()
        total_loss = 0.0
        n_batches = 0
        n_correct = 0
        n_seen = 0
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            x = binarize(imgs, thresholds=thresholds, flatten=flatten)
            out = model(x, hard=False, sharpness=sharpness, ste=args.ste)
            loss = loss_fn(out, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
            n_correct += (out.argmax(dim=-1) == labels).sum().item()
            n_seen += labels.numel()

        avg_loss = total_loss / max(n_batches, 1)
        elapsed = time.time() - t0

        train_acc = n_correct / max(n_seen, 1)
        log_entry = dict(epoch=epoch, loss=avg_loss, train_acc=train_acc, sharpness=sharpness,
                          weight_decay=wd_now, elapsed=elapsed)

        msg = (f"epoch {epoch} loss={avg_loss:.4f} train_acc={train_acc:.4f} sharpness={sharpness:.2f} "
               f"weight_decay={wd_now:.5f} elapsed={elapsed:.1f}s")

        is_eval_epoch = (epoch % args.eval_every == 0) or (epoch == args.epochs - 1)
        if is_eval_epoch:
            val_acc = evaluate(model, val_loader, thresholds, device, hard=False, sharpness=1.0, flatten=flatten)
            val_hard_acc = evaluate(model, val_loader, thresholds, device, hard=True, flatten=flatten)
            confs = model.gate_confidences()
            mean_conf = sum(confs.values()) / len(confs)
            log_entry.update(val_acc=val_acc, val_hard_acc=val_hard_acc, mean_gate_confidence=mean_conf)
            msg += f" val_acc={val_acc:.4f} val_hard_acc={val_hard_acc:.4f} gate_conf={mean_conf:.3f}"

            state = dict(
                model=model.state_dict(), opt=opt.state_dict(), epoch=epoch,
                best_val=best_val, arch=expected_arch,
            )
            torch.save(state, ckpt_path)
            if val_hard_acc > best_val:
                best_val = val_hard_acc
                state["best_val"] = best_val
                torch.save(state, str(ckpt_path) + ".best")

        print(msg)
        log_f.write(json.dumps(log_entry) + "\n")
        log_f.flush()

    log_f.close()

    # Final eval: prefer the best checkpoint if present.
    best_path = Path(str(ckpt_path) + ".best")
    if best_path.exists():
        state = torch.load(best_path, map_location=device)
        model.load_state_dict(state["model"])
        print(f"Loaded best checkpoint (epoch {state['epoch']}, best_val={state.get('best_val')}) for final eval.")

    soft_acc = evaluate(model, test_loader, thresholds, device, hard=False, sharpness=1.0, flatten=flatten)
    hard_acc = evaluate(model, test_loader, thresholds, device, hard=True, flatten=flatten)
    confs = model.gate_confidences()
    mean_conf = sum(confs.values()) / len(confs)
    gap = soft_acc - hard_acc
    print(f"FINAL test soft_acc(t=1)={soft_acc:.4f} hard_acc={hard_acc:.4f} "
          f"mean_gate_confidence={mean_conf:.3f} soft/hard gap={gap:.4f}")
    with open(log_path, "a") as f:
        f.write(json.dumps(dict(final=True, test_soft_acc=soft_acc, test_hard_acc=hard_acc,
                                gap=gap, mean_gate_confidence=mean_conf, ste=args.ste,
                                init_from=args.init_from)) + "\n")


if __name__ == "__main__":
    main()


## Run tests

In [ ]:
!python test_difflogic.py
# quick GPU check: one soft + hard forward/backward on a tiny LogicTreeNet
import torch
from difflogic import LogicTreeNet, binarize
dev = "cuda" if torch.cuda.is_available() else "cpu"
m = LogicTreeNet(in_channels=9, in_size=32, channels=(4, 8, 16, 32), num_classes=10, tau=10.0).to(dev)
xb = binarize(torch.rand(8, 3, 32, 32, device=dev), (0.25, 0.5, 0.75), flatten=False)
m(xb).sum().backward(); m(xb, hard=True)
print("GPU check passed on", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

## Checkpoint storage (Kaggle working directory)

Kaggle notebooks have no `google.colab.drive` — checkpoints go to `/kaggle/working`,
which persists only for the interactive session unless you click
**Save Version -> Save & Run All (Commit)** before you close the tab.

In [ ]:
import os

CKPT_DIR = "/kaggle/working/pbnn_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)


## Resume earlier stages (if any)

In [ ]:
import glob, os, shutil

# resume any stage that already ran (attach a previous version's output under Input)
for stem in ["conv_S_ctrl", "conv_S_ft_ste", "conv_S_ft_norm"]:
    for pattern, name in [(f"{stem}_ckpt*.pt.best", f"{stem}_ckpt.pt.best"),
                          (f"{stem}_ckpt*.pt", f"{stem}_ckpt.pt"),
                          (f"{stem}_log*.jsonl", f"{stem}_log.jsonl")]:
        found = [h for h in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
                 if not (name.endswith(".pt") and h.endswith(".best"))]
        if found:
            shutil.copy(found[0], f"{CKPT_DIR}/{name}")
            print(f"resuming {stem}: imported {found[0]}")
INIT = f"{CKPT_DIR}/conv_S_ctrl_ckpt.pt.best"


## Stage 1: train the control baseline (~5.5 h)

In [ ]:
!python train.py --model conv --scale S --tree-depth 3 --kernel-size 3 \
    --epochs 200 --batch-size 128 --optimizer adamw --lr 0.02 --weight-decay 0.002 \
    --thresholds 0.25,0.5,0.75 --val-split 5000 --eval-every 5 \
    --ckpt "{CKPT_DIR}/conv_S_ctrl_ckpt.pt" --log "{CKPT_DIR}/conv_S_ctrl_log.jsonl"


## Arm A: straight-through fine-tune (~35-40 min)

In [ ]:
!python train.py --model conv --scale S --tree-depth 3 --kernel-size 3 --epochs 20 --batch-size 128 --optimizer adamw --lr 0.005 --weight-decay 0 --thresholds 0.25,0.5,0.75 --val-split 5000 --eval-every 1 --init-from "{INIT}" --ste \
    --ckpt "{CKPT_DIR}/conv_S_ft_ste_ckpt.pt" --log "{CKPT_DIR}/conv_S_ft_ste_log.jsonl"


## Arm B: normal fine-tune, same settings (~35-40 min)

In [ ]:
!python train.py --model conv --scale S --tree-depth 3 --kernel-size 3 --epochs 20 --batch-size 128 --optimizer adamw --lr 0.005 --weight-decay 0 --thresholds 0.25,0.5,0.75 --val-split 5000 --eval-every 1 --init-from "{INIT}" \
    --ckpt "{CKPT_DIR}/conv_S_ft_norm_ckpt.pt" --log "{CKPT_DIR}/conv_S_ft_norm_log.jsonl"


## Compare

In [ ]:
import json
import matplotlib.pyplot as plt

def load(path):
    rows = [json.loads(l) for l in open(path) if l.strip()]
    return [r for r in rows if "epoch" in r], next((r for r in reversed(rows) if r.get("final")), None)

arms = {"A: straight-through": f"{CKPT_DIR}/conv_S_ft_ste_log.jsonl",
        "B: normal": f"{CKPT_DIR}/conv_S_ft_norm_log.jsonl"}
print(f"{'':24s}{'test soft':>10s}{'test hard':>11s}{'gap':>8s}")
print(f"{'control (run 7)':24s}{57.69:>9.2f}%{50.56:>10.2f}%{7.13:>8.2f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, path in arms.items():
    try:
        rows, final = load(path)
    except FileNotFoundError:
        print(f"{name:24s}  (not run yet)"); continue
    if final:
        print(f"{name:24s}{100*final['test_soft_acc']:>9.2f}%{100*final['test_hard_acc']:>10.2f}%"
              f"{100*final['gap']:>8.2f}")
    ep = [r["epoch"] for r in rows if "val_hard_acc" in r]
    ax[0].plot(ep, [r["val_hard_acc"] for r in rows if "val_hard_acc" in r], marker="o", label=name)
    ax[1].plot(ep, [r["val_acc"] for r in rows if "val_acc" in r], marker="o", label=name)
for a, t in zip(ax, ["validation HARD accuracy", "validation soft accuracy"]):
    a.axhline(0.5082 if "HARD" in t else 0.5712, color="gray", ls="--", label="control")
    a.set_title(t); a.set_xlabel("fine-tune epoch"); a.legend()
plt.tight_layout(); plt.show()
print("\nCompare the HARD column: A beats B by more than ~3 points -> the straight-through fine-tune helps.")
print("(A's soft accuracy dropping is expected: it trains the discretized network directly.)")


## When you're done

Send the notebook back with its outputs. Before closing, click **Save Version** so the
checkpoints in `/kaggle/working/pbnn_checkpoints` are kept. If the session stops partway,
upload the `conv_S_ft_*` files as a dataset, attach it with the control checkpoint, and
Run All: finished arms skip straight to their final evaluation and unfinished ones resume.